In [1]:
import pandas as pd
import pyreadstat
import sys
import os

pwd = os.getcwd()
mappings_dir = os.path.join(pwd, '..', 'src', 'atc-ndc-bidirectional-conversion', 'mappings')

sys.path.append(mappings_dir)
sys.path.insert(0, str(mappings_dir))
from lookup_code import lookup_code

path = "/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_pin_20211105.sas7bdat"
pin_df, meta = pyreadstat.read_sas7bdat(path, output_format="pandas")
print (pin_df.columns)
print (pin_df.head())

ATC_codes = pin_df['SUPP_DRUG_ATC_CODE'].unique()

/home/weijiesun/anaconda3/envs/pytorch/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Index(['PATID', 'DSPN_DATE', 'DSPN_AMT_QTY', 'DSPN_AMT_UNT_MSR_CD',
       'DSPN_DAY_SUPPLY_QTY', 'DRUG_DIN', 'SUPP_DRUG_ATC_CODE'],
      dtype='object')
   PATID  DSPN_DATE  DSPN_AMT_QTY DSPN_AMT_UNT_MSR_CD  DSPN_DAY_SUPPLY_QTY  \
0    1.0 2008-04-01          60.0                 TAB                 30.0   
1    1.0 2008-04-01          60.0                 TAB                 30.0   
2    1.0 2008-04-01          60.0                 TAB                 30.0   
3    1.0 2008-04-01          60.0                 TAB                 30.0   
4    1.0 2008-04-18          30.0                 TAB                 30.0   

   DRUG_DIN SUPP_DRUG_ATC_CODE  
0  02212331            A02BA02  
1  02242681            B01AA03  
2  02242681            B01AA03  
3  02212331            A02BA02  
4  02010909            G04CB01  


In [1]:
import pandas as pd
import pyreadstat
import sys
import os
# path = "/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_dad_20211105.pickle"
# ed_df = pd.read_pickle(path)
# print (ed_df.head())
# print (ed_df.columns)

In [6]:
# parse datetimes (won't overwrite original string columns)
pre_df['starttime_dt'] = pd.to_datetime(pre_df['starttime'], errors='coerce')
pre_df['stoptime_dt'] = pd.to_datetime(pre_df['stoptime'], errors='coerce')

# count where start > stop
mask = pre_df['starttime_dt'] > pre_df['stoptime_dt']
count = int(mask.sum())
total = len(pre_df)
print(f"Rows with starttime > stoptime: {count} of {total} ({count/total:.4%})")

# also report parsing failures
n_start_na = pre_df['starttime_dt'].isna().sum()
n_stop_na = pre_df['stoptime_dt'].isna().sum()
print(f"Unparsed starttime: {n_start_na}, unparsed stoptime: {n_stop_na}")

# show a few examples
pre_df.loc[mask, ['subject_id','hadm_id','starttime','stoptime']].head(10)

Rows with starttime > stoptime: 816994 of 20292611 (4.0261%)
Unparsed starttime: 21890, unparsed stoptime: 31436


,subject_id,hadm_id,starttime,stoptime
0,10000032,22595853,2180-05-08 08:00:00,2180-05-07 22:00:00
6,10000032,22595853,2180-05-08 08:00:00,2180-05-07 22:00:00
19,10000032,22841357,2180-06-26 23:00:00,2180-06-26 22:00:00
42,10000032,25742920,2180-08-06 03:00:00,2180-08-06 02:00:00
83,10000084,23052089,2160-11-24 15:00:00,2160-11-24 14:00:00
188,10000690,23280645,2150-09-24 10:00:00,2150-09-23 19:00:00
223,10000690,25860671,2150-11-09 08:00:00,2150-11-08 10:00:00
224,10000690,25860671,2150-11-09 08:00:00,2150-11-08 10:00:00
230,10000690,25860671,2150-11-03 01:00:00,2150-11-03 00:00:00
234,10000690,25860671,2150-11-03 00:00:00,2150-11-02 23:00:00


In [8]:
# 然后我想check 有多少 starttime_dt 不等于 stoptime_dt
neq_mask = pre_df['starttime_dt'].ne(pre_df['stoptime_dt'])
# only count rows where both datetimes parsed (exclude rows with NaT)
neq_mask_valid = neq_mask & pre_df['starttime_dt'].notna() & pre_df['stoptime_dt'].notna()
# 比较仅日期（忽略时间）
start_date = pre_df['starttime_dt'].dt.normalize()
stop_date = pre_df['stoptime_dt'].dt.normalize()

neq_date_mask = start_date.ne(stop_date)
neq_date_mask_valid = neq_date_mask & pre_df['starttime_dt'].notna() & pre_df['stoptime_dt'].notna()

n_neq_dates = int(neq_date_mask_valid.sum())
print(f"Rows with different date (ignoring time): {n_neq_dates} of {len(pre_df):,} ({n_neq_dates/len(pre_df):.4%})")

# 显示示例
pre_df.loc[neq_date_mask_valid, ['subject_id','hadm_id','starttime','stoptime','starttime_dt','stoptime_dt']].head(20)
n_neq = int(neq_mask_valid.sum())
print(f"Rows with starttime_dt != stoptime_dt (both parsed): {n_neq} of {len(pre_df):,} ({n_neq/len(pre_df):.4%})")

# also report rows where one is missing and the other is not
one_missing = pre_df['starttime_dt'].isna() ^ pre_df['stoptime_dt'].isna()
n_one_missing = int(one_missing.sum())
print(f"Rows where exactly one of starttime_dt/stoptime_dt is missing: {n_one_missing:,}")

# show examples
pre_df.loc[neq_mask_valid, ['subject_id','hadm_id','starttime','stoptime','starttime_dt','stoptime_dt']].head(20)

Rows with different date (ignoring time): 15936285 of 20,292,611 (78.5325%)
Rows with starttime_dt != stoptime_dt (both parsed): 19822696 of 20,292,611 (97.6843%)
Rows where exactly one of starttime_dt/stoptime_dt is missing: 11,366


,subject_id,hadm_id,starttime,stoptime,starttime_dt,stoptime_dt
0,10000032,22595853,2180-05-08 08:00:00,2180-05-07 22:00:00,2180-05-08 08:00:00,2180-05-07 22:00:00
1,10000032,22595853,2180-05-07 02:00:00,2180-05-07 22:00:00,2180-05-07 02:00:00,2180-05-07 22:00:00
2,10000032,22595853,2180-05-07 01:00:00,2180-05-07 09:00:00,2180-05-07 01:00:00,2180-05-07 09:00:00
4,10000032,22595853,2180-05-07 00:00:00,2180-05-07 22:00:00,2180-05-07 00:00:00,2180-05-07 22:00:00
5,10000032,22595853,2180-05-07 01:00:00,2180-05-07 22:00:00,2180-05-07 01:00:00,2180-05-07 22:00:00
6,10000032,22595853,2180-05-08 08:00:00,2180-05-07 22:00:00,2180-05-08 08:00:00,2180-05-07 22:00:00
7,10000032,22595853,2180-05-07 01:00:00,2180-05-07 22:00:00,2180-05-07 01:00:00,2180-05-07 22:00:00
8,10000032,22595853,2180-05-07 00:00:00,2180-05-07 22:00:00,2180-05-07 00:00:00,2180-05-07 22:00:00
9,10000032,22595853,2180-05-07 01:00:00,2180-05-07 22:00:00,2180-05-07 01:00:00,2180-05-07 22:00:00
10,10000032,22595853,2180-05-07 01:00:00,2180-05-07 22:00:00,2180-05-07 01:00:00,2180-05-07 22:00:00


In [33]:
proc_cols = [c for c in dad_df.columns if c.lower().startswith('proc')]
print(proc_cols)
print(f"Count: {len(proc_cols)}")

# 若要查看这些列的前几行：
dad_df[proc_cols].head()

['PROCCODE1', 'PROCCODE2', 'PROCCODE3', 'PROCCODE4', 'PROCCODE5', 'PROCCODE6', 'PROCCODE7', 'PROCCODE8', 'PROCCODE9', 'PROCCODE10', 'PROCCODE11', 'PROCCODE12', 'PROCCODE13', 'PROCCODE14', 'PROCCODE15', 'PROCCODE16', 'PROCCODE17', 'PROCCODE18', 'PROCCODE19', 'PROCCODE20', 'PROCSVC1', 'PROCSVC2', 'PROCSVC3', 'PROCSVC4', 'PROCSVC5', 'PROCSVC6', 'PROCSVC7', 'PROCSVC8', 'PROCSVC9', 'PROCSVC10', 'PROCSVC11', 'PROCSVC12', 'PROCSVC13', 'PROCSVC14', 'PROCSVC15', 'PROCSVC16', 'PROCSVC17', 'PROCSVC18', 'PROCSVC19', 'PROCSVC20', 'PROCSTDT1_DT', 'PROCSTDT2_DT', 'PROCSTDT3_DT', 'PROCSTDT4_DT', 'PROCSTDT5_DT', 'PROCSTDT6_DT', 'PROCSTDT7_DT', 'PROCSTDT8_DT', 'PROCSTDT9_DT', 'PROCSTDT10_DT', 'PROCSTDT11_DT', 'PROCSTDT12_DT', 'PROCSTDT13_DT', 'PROCSTDT14_DT', 'PROCSTDT15_DT', 'PROCSTDT16_DT', 'PROCSTDT17_DT', 'PROCSTDT18_DT', 'PROCSTDT19_DT', 'PROCSTDT20_DT', 'PROCST1_DTTM', 'PROCST2_DTTM', 'PROCST3_DTTM', 'PROCST4_DTTM', 'PROCST5_DTTM', 'PROCST6_DTTM', 'PROCST7_DTTM', 'PROCST8_DTTM', 'PROCST9_DTTM', 'P

,PROCCODE1,PROCCODE2,PROCCODE3,PROCCODE4,PROCCODE5,PROCCODE6,PROCCODE7,PROCCODE8,PROCCODE9,PROCCODE10,...,PROCE11_DTTM,PROCE12_DTTM,PROCE13_DTTM,PROCE14_DTTM,PROCE15_DTTM,PROCE16_DTTM,PROCE17_DTTM,PROCE18_DTTM,PROCE19_DTTM,PROCE20_DTTM
0,,,,,,,,,,,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
1,3SC10VA,3SQ10VA,,,,,,,,,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
2,1NQ89SFXXG,1NK77RR,,,,,,,,,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
3,,,,,,,,,,,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
4,2PC71HA,,,,,,,,,,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT


In [6]:
# I want to save All ATC_codes into a text file, also privide how to read it.
with open('/data/padmalab_external/special_project/AHS_Data_Release_2/ATC_codes.txt', 'w') as f:
    for code in ATC_codes:
        f.write(f"{code}\n")
ATC_codes = []
with open('/data/padmalab_external/special_project/AHS_Data_Release_2/ATC_codes.txt', 'r') as f:
    ATC_codes = f.read().splitlines()

In [2]:

import tqdm
print (f"Total unique ATC codes: {len(ATC_codes)}") 
count = 0

for code in tqdm.tqdm(ATC_codes):
    result = lookup_code(code)
    if 'not found' not in result:
        count += 1
print (f"Found {count} out of {len(ATC_codes)} ATC codes")

Total unique ATC codes: 1713


  0%|          | 0/1713 [00:00<?, ?it/s]

100%|██████████| 1713/1713 [22:24<00:00,  1.27it/s]

Found 129 out of 1713 ATC codes


In [ ]:
import pandas as pd
import pyreadstat

# path = "/data/padmalab_external/special_project/physionet.org/files/mimiciv/3.1/hosp/prescriptions.csv.gz"
prescriptions_df = pd.read_csv(path, compression='gzip', low_memory=False)
print (prescriptions_df.columns)
print (prescriptions_df.head())

Index(['subject_id', 'hadm_id', 'pharmacy_id', 'poe_id', 'poe_seq',
       'order_provider_id', 'starttime', 'stoptime', 'drug_type', 'drug',
       'formulary_drug_cd', 'gsn', 'ndc', 'prod_strength', 'form_rx',
       'dose_val_rx', 'dose_unit_rx', 'form_val_disp', 'form_unit_disp',
       'doses_per_24_hrs', 'route'],
      dtype='object')
   subject_id   hadm_id  pharmacy_id       poe_id  poe_seq order_provider_id  \
0    10000032  22595853     12775705  10000032-55     55.0            P85UQ1   
1    10000032  22595853     18415984  10000032-42     42.0            P23SJA   
2    10000032  22595853     23637373  10000032-35     35.0            P23SJA   
3    10000032  22595853     26862314  10000032-41     41.0            P23SJA   
4    10000032  22595853     30740602  10000032-27     27.0            P23SJA   

             starttime             stoptime drug_type  \
0  2180-05-08 08:00:00  2180-05-07 22:00:00      MAIN   
1  2180-05-07 02:00:00  2180-05-07 22:00:00      MAIN   
2  2

In [5]:
prescriptions_df['ndc']
# convert 51079007220 to 51079-0072-20 if 10 characters,
# if 8 characters, convert 51079072 to 51079-072

def ndc_to_11_hyphen(s: str) -> str:
    digits = ''.join(ch for ch in s if ch.isdigit())
    ndc11 = digits.zfill(11)[:11]  # 左填充到 11 位
    return f"{ndc11[:5]}-{ndc11[5:9]}-{ndc11[9:]}"
print (len(prescriptions_df['ndc'].unique()))
unique_ndcs = prescriptions_df['ndc'].unique()
for i in tqdm.tqdm(range(len(unique_ndcs))):
    # test_ndc = prescriptions_df['ndc'].unique()[i]
    # print (test_ndc, "->", ndc_to_11_hyphen(str(test_ndc)))
    ndc = unique_ndcs[i]
    ndc = ndc_to_11_hyphen(str(ndc))
    result = lookup_code(ndc)
    if 'not found' not in result:
        count += 1
print (f"Found {count} out of {len(unique_ndcs)} NDC codes")

6589


100%|██████████| 6589/6589 [1:27:40<00:00,  1.25it/s]

Found 1873 out of 6589 NDC codes


In [ ]:
path = "/data/padmalab_external/special_project/meds_pipeline_output/mimic/mimic_meds_core.parquet"
meds_core_df = pd.read_parquet(path)
print (meds_core_df.columns)

Index(['subject_id', 'time', 'event_type', 'code', 'code_system',
       'encounter_id', 'value_num', 'unit', 'value_text', 'source_table',
       'provenance_id'],
      dtype='object')


In [10]:
ls /data/padmalab_external/special_project/MIMIC_IV/

files/  MEDS_cohort/  meds_etl/  pre_MEDS/  raw_input/  robots.txt*


In [11]:
# I want to save All ATC_codes into a text file, also privide how to read it.
with open('/data/padmalab_external/special_project/MIMIC_IV/NDC_codes.txt', 'w') as f:
    for code in unique_ndcs:
        f.write(f"{code}\n")
unique_ndcs = []
with open('/data/padmalab_external/special_project/MIMIC_IV/NDC_codes.txt', 'r') as f:
    unique_ndcs = f.read().splitlines()
print (len(unique_ndcs))
print (unique_ndcs[:10])

6589
['51079007320.0', '487980125.0', '51079007220.0', '245004101.0', '0.0', '6022761.0', '63739054410.0', '904198861.0', '19515089452.0', '173068224.0']


In [ ]:
import sys
import os

pwd = os.getcwd()
mappings_dir = os.path.join(pwd, '..', 'src', 'atc-ndc-bidirectional-conversion', 'mappings')

sys.path.append(mappings_dir)

sys.path.insert(0, str(mappings_dir))
from lookup_code import lookup_code

In [37]:
result = lookup_code("B01AA03")
result
# 
# 47335-0985-60"

"❌ ATC code 'B01AA03' not found in database"

In [ ]:
import pandas as pd
import os

path = "/data/padmalab_external/special_project/meds_pipeline_output/mimic/mimic_meds_core.parquet"

# overwrites meds_core_df if it exists
if os.path.exists(path):
    meds_core_df = pd.read_parquet(path)  # add engine='pyarrow' or engine='fastparquet' if needed
    print("Loaded:", path)
    print("shape:", meds_core_df.shape)
    print("columns:", list(meds_core_df.columns))
    meds_core_df.head()
else:
    print("File not found:", path)

Loaded: /data/padmalab_external/special_project/meds_pipeline_output/mimic/mimic_meds_plus.parquet
shape: (4474, 14)
columns: ['subject_id', 'time', 'event_type', 'code', 'diagnosis_source', 'hadm_id', 'stay_id', 'seq_num', 'diagnosis_description', 'discharge_time', 'icd_version', 'value_text', 'source_table', 'provenance_id']


In [2]:
# quick inspect
import pandas as pd, yaml
cfg = yaml.safe_load(open("../src/meds_pipeline/configs/mimic.yaml"))
raw = cfg["raw_paths"]
ed_diag = pd.read_csv(raw["ed_diagnoses_icd"], compression="gzip", low_memory=False)
ed_stays = pd.read_csv(raw["ed"], compression="gzip", low_memory=False)

print("ed_diag cols:", ed_diag.columns.tolist())
print("ed_stays cols:", ed_stays.columns.tolist())
print("ed_diag dtypes:", ed_diag[ed_diag.columns.intersection(['stay_id','subject_id'])].dtypes.to_dict())
print("ed_stays dtypes:", ed_stays[ed_stays.columns.intersection(['stay_id','subject_id'])].dtypes.to_dict())

print("sample stay_ids diag:", ed_diag['stay_id'].dropna().astype(str).head(10).tolist())
print("sample stay_ids stays:", ed_stays['stay_id'].dropna().astype(str).head(10).tolist())
print("overlap (direct):", ed_diag['stay_id'].isin(ed_stays['stay_id']).sum(), "of", ed_diag['stay_id'].nunique())

ed_diag cols: ['subject_id', 'stay_id', 'seq_num', 'icd_code', 'icd_version', 'icd_title']
ed_stays cols: ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'gender', 'race', 'arrival_transport', 'disposition']
ed_diag dtypes: {'subject_id': dtype('int64'), 'stay_id': dtype('int64')}
ed_stays dtypes: {'subject_id': dtype('int64'), 'stay_id': dtype('int64')}
sample stay_ids diag: ['32952584', '32952584', '32952584', '33258284', '33258284', '33258284', '33258284', '35968195', '35968195', '35968195']
sample stay_ids stays: ['33258284', '38112554', '35968195', '32952584', '39399961', '35203156', '36954971', '36533795', '32522732', '39513268']
overlap (direct): 899050 of 423989


In [21]:
import pandas as pd
import os

# ls /data/padmalab_external/special_project/MIMIC_IV/MEDS_cohort/metadata/codes.parquet
path = "/data/padmalab_external/special_project/MIMIC_IV/MEDS_cohort/data/train/0.parquet"
# path = "/data/padmalab_external/special_project/MIMIC_IV/MEDS_cohort/metadata/codes.parquet"
df = pd.read_parquet(path)
df

,subject_id,time,code,numeric_value,insurance,language,marital_status,race,hadm_id,drg_severity,...,route,frequency,doses_per_24_hrs,poe_id,icustay_id,order_id,link_order_id,unit,ordercategorydescription,statusdescription
0,10007275,NaT,GENDER//M,NaN,None,None,None,None,NaN,NaN,...,None,None,NaN,None,NaN,NaN,NaN,None,None,None
1,10007275,2067-01-01 00:00:00,MEDS_BIRTH,NaN,None,None,None,None,NaN,NaN,...,None,None,NaN,None,NaN,NaN,NaN,None,None,None
2,10007275,2117-09-17 20:59:00,TRANSFER_TO//ED//Emergency Department,NaN,None,None,None,None,NaN,NaN,...,None,None,NaN,None,NaN,NaN,NaN,None,None,None
3,10011873,NaT,GENDER//F,NaN,None,None,None,None,NaN,NaN,...,None,None,NaN,None,NaN,NaN,NaN,None,None,None
4,10011873,2064-01-01 00:00:00,MEDS_BIRTH,NaN,None,None,None,None,NaN,NaN,...,None,None,NaN,None,NaN,NaN,NaN,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1968619,19997576,2190-11-27 00:00:00,Blood Pressure,NaN,None,None,None,None,NaN,NaN,...,None,None,NaN,None,NaN,NaN,NaN,None,None,None
1968620,19998972,NaT,GENDER//M,NaN,None,None,None,None,NaN,NaN,...,None,None,NaN,None,NaN,NaN,NaN,None,None,None
1968621,19998972,2069-01-01 00:00:00,MEDS_BIRTH,NaN,None,None,None,None,NaN,NaN,...,None,None,NaN,None,NaN,NaN,NaN,None,None,None
1968622,19998972,2124-04-22 14:02:00,TRANSFER_TO//ED//Emergency Department,NaN,None,None,None,None,NaN,NaN,...,None,None,NaN,None,NaN,NaN,NaN,None,None,None


In [30]:
ls /data/padmalab_external/special_project/physionet.org/files/ptb-xl/1.0.3/records500/

00000/  03000/  06000/  09000/  12000/  15000/  18000/  21000/
01000/  04000/  07000/  10000/  13000/  16000/  19000/  index.html*
02000/  05000/  08000/  11000/  14000/  17000/  20000/


In [17]:
# 统计 code 字段中 '//' 之前的类型并展示计数与示例
prefix = df['code'].astype(str).str.split('//').str[0].fillna('')
counts = prefix.value_counts()

print("Types and counts:")
print(counts.to_string())

print(f"\n总共不同类型: {len(counts)}\n")

# 每种类型展示最多 3 条示例（如需更多可调整 n）
n = 3
for t in counts.index:
    print(f"\n== {t} ({counts[t]} rows) ==")
    display(df[df['code'].str.startswith(t + '//')].head(n)[['code', 'description']])

Types and counts:
code
DIAGNOSIS               28583
PROCEDURE               15137
LAB                       895
INFUSION_START            174
INFUSION_END              174
HCPCS                      71
SUBJECT_FLUID_OUTPUT       71

总共不同类型: 7


== DIAGNOSIS (28583 rows) ==


,code,description
0,DIAGNOSIS//ICD//10//M5011,"Cervical disc disorder with radiculopathy, hig..."
2,DIAGNOSIS//ICD//10//S32431A,Displaced fracture of anterior column [iliopub...
3,DIAGNOSIS//ICD//10//T7401XA,"Adult neglect or abandonment, confirmed, initi..."



== PROCEDURE (15137 rows) ==


,code,description
1,PROCEDURE//ICD//10//0XH933Z,Insertion of Infusion Device into Left Upper A...
6,PROCEDURE//ICD//10//B32GYZZ,Computerized Tomography (CT Scan) of Bilateral...
12,PROCEDURE//ICD//10//08RN07Z,Replacement of Right Upper Eyelid with Autolog...



== LAB (895 rows) ==


...

,code,description
375,INFUSION_END//229760,epoprostenol Injection [Veletri]
767,INFUSION_END//221456,calcium gluconate Injection
770,INFUSION_END//227526,sodium citrate Injectable Solution



== HCPCS (71 rows) ==


,code,description
93,HCPCS//Pacemaker – Leadless and Pocketless System,
1916,HCPCS//Mediastinum and diaphragm,
2115,HCPCS//Lap esophagomyotomy,Laparoscopic esophagomyotomy (heller type)



== SUBJECT_FLUID_OUTPUT (71 rows) ==


,code,description
866,SUBJECT_FLUID_OUTPUT//226603//mL,Fluid output biliary drain
1550,SUBJECT_FLUID_OUTPUT//227511//mL,TF Residual Output
1938,SUBJECT_FLUID_OUTPUT//226633//mL,Pre-Admission


In [1]:
ls /data/padmalab_external/special_project/AHS_Data_Release_2/

 2M_xml_statmenet_code_one_hot.pickle
 AHS_Data_release_2_ECG_study_data_splits.pickle
 all_ecg_patid_list.pickle
 ATC_codes.txt
'DXL_ECG_Algorithm_Physician_s_Guide_(ENG)_Ed.2.pdf'
 ecg_np/
 episode_creation/
 external_ecg_patid_list.pickle
 Globalmeasurements.pickle
 Mortailty_ResNet_Cox_test.pkl
 Mortailty_ResNet_Cox_train.pkl
 rmt22884_amb_20211105.sas7bdat*
 rmt22884_claims_20211105.sas7bdat
 rmt22884_dad_20211105.pickle
 rmt22884_dad_20211105.sas7bdat*
 rmt22884_dad_20211105_w_episode_order.pickle
 rmt22884_dad_proc_20250422.sas7bdat*
 rmt22884_dad_proc_20250429.sas7bdat*
 rmt22884_death_20251124.parquet
 rmt22884_death_label_20211105.pickle
 rmt22884_demographics.pkl
 rmt22884_demographics_with_death.pkl
 rmt22884_ecg_20211105_df.pickle*
 rmt22884_ECG_xml_20211105.pickle
 rmt22884_ed_20211105.pickle
 rmt22884_ed_20211105_w_episode_order.pickle
 rmt22884_episode_df_all.pickle*
 rmt22884_lab_2012_2016_20211105.sas7bdat
 rmt22884_lab_2017_2021_20211105.sas7bdat
'RMT 22884 - List of

In [2]:
import pandas as pd
import pyreadstat

# amb_df, _ = pyreadstat.read_sas7bdat("/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_amb_20211105.sas7bdat", output_format="pandas")
# amb_df

ed_df = pd.read_pickle('/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_ed_20211105.pickle')
ed_df.head()

,PATID,VISIT_DATE_DT,DXCODE1,DXCODE2,DXCODE3,DXCODE4,DXCODE5,DXCODE6,DXCODE7,DXCODE8,...,PROCCODE1,PROCCODE2,PROCCODE3,PROCCODE4,PROCCODE5,PROCCODE6,PROCCODE7,PROCCODE8,PROCCODE9,PROCCODE10
0,46,2021-05-24,R53,,,,,,,,...,,,,,,,,,,
1,128,2021-05-19,F151,F155,,,,,,,...,,,,,,,,,,
4,572,2021-05-23,H609,,,,,,,,...,,,,,,,,,,
7,1072,2021-05-22,J9691,I500,E875,,,,,,...,3GY10VA,3IP30DD,,,,,,,,
9,1232,2021-05-25,R1012,E1023,N0839,,,,,,...,3OT10VA,3GY10VA,,,,,,,,


In [3]:
ed_df['ADMITBYAMB'].value_counts()

ADMITBYAMB
N    2589037
G     907726
      223954
C       8262
A       5812
Name: count, dtype: int64

In [4]:
dad_df = pd.read_pickle('/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_dad_20211105.pickle')
dad_df.head()

,PATID,ADMITDATE_DT,DISDATE_DT,AGE_ADMIT,SEX,DXCODE1,DXCODE2,DXCODE3,DXCODE4,DXCODE5,...,PROCSVC11,PROCSVC12,PROCSVC13,PROCSVC14,PROCSVC15,PROCSVC16,PROCSVC17,PROCSVC18,PROCSVC19,PROCSVC20
0,1343,2021-08-19,2021-09-04,83.0,M,M1000,M130,M6593,M1196,Z751,...,,,,,,,,,,
1,2309,2021-08-18,2021-08-20,57.0,M,T825,Y712,,,,...,,,,,,,,,,
2,6157,2021-08-31,2021-09-01,79.0,M,T828,Y831,,,,...,,,,,,,,,,
3,7550,2021-08-13,2021-08-20,30.0,M,E840,E848,,,,...,,,,,,,,,,
4,11862,2021-08-22,2021-08-27,44.0,M,A099,K6388,N179,E1028,E1023,...,,,,,,,,,,


In [5]:
dad_df['ADMITCAT'].value_counts()

ADMITCAT
U    915040
L    262826
N      7505
Name: count, dtype: int64

In [ ]:
dad_df = pd.read_pickle('/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_ed_20211105.pickle')
ed_df.head()

In [1]:
import pandas as pd
import pyreadstat

registry_df, _ = pyreadstat.read_sas7bdat("/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_reg_20211105.sas7bdat", output_format="pandas")
registry_df

/home/weijiesun/anaconda3/envs/pytorch/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


,PATID,FYE,ACTIVE_COVERAGE,DEATH_IND,SEX,IN_MIGRATION_IND,OUT_MIGRATION_IND,PERS_REAP_END_DATE,PERS_REAP_END_RSN_CODE_GRP,Birth_Year
0,1.0,2007.0,1,0,M,0,0,NaT,N/A,1949.0
1,1.0,2008.0,1,0,M,0,0,NaT,N/A,1949.0
2,1.0,2009.0,1,0,M,0,0,NaT,N/A,1949.0
3,1.0,2010.0,1,0,M,0,0,NaT,N/A,1949.0
4,1.0,2011.0,1,0,M,0,0,NaT,N/A,1949.0
...,...,...,...,...,...,...,...,...,...,...
3228645,295800.0,2017.0,1,0,M,0,0,NaT,N/A,2014.0
3228646,295800.0,2018.0,1,0,M,0,0,NaT,N/A,2014.0
3228647,295800.0,2019.0,1,0,M,0,0,NaT,N/A,2014.0
3228648,295800.0,2020.0,1,0,M,0,0,NaT,N/A,2014.0


In [5]:
import pandas as pd

path = "/data/padmalab_external/special_project/meds_pipeline_output/mimic/mimic_meds_core.parquet"
# path = "/data/padmalab_external/special_project/meds_pipeline_output/ahs/ahs_meds_core.parquet"
test_df = pd.read_parquet(path)
test_df.head()

,subject_id,time,event_type,code,code_system,diagnosis_source,value,encounter_id,value_num,unit,value_text,source_table,provenance_id
0,10000032,2180-05-06 22:23:00,encounter.start,ADMIT//HOSP//URGENT,EVENT,None,<NA>,None,NaN,None,None,None,None
1,10000032,2180-06-26 18:27:00,encounter.start,ADMIT//HOSP//EW EMER.,EVENT,None,<NA>,None,NaN,None,None,None,None
2,10000032,2180-08-05 23:44:00,encounter.start,ADMIT//HOSP//EW EMER.,EVENT,None,<NA>,None,NaN,None,None,None,None
3,10000032,2180-07-23 12:35:00,encounter.start,ADMIT//HOSP//EW EMER.,EVENT,None,<NA>,None,NaN,None,None,None,None
4,10000068,2160-03-03 23:16:00,encounter.start,ADMIT//HOSP//EU OBSERVATION,EVENT,None,<NA>,None,NaN,None,None,None,None


In [23]:
test_df['event_type'].unique()

array(['encounter.start', 'encounter.end', 'ed.entry', 'ed.exit',
       'diagnosis', 'procedures', 'medication.order',
       'demographics.birth', 'demographics.sex', 'death'], dtype=object)

In [28]:
test_df[test_df['event_type'] == 'encounter.end']


,subject_id,time,event_type,code,code_system,diagnosis_source,value,encounter_id,value_num,unit,value_text,source_table,provenance_id
303,10000032,2180-05-07 17:15:00,encounter.end,DISCHARGE//HOSP//HOME,EVENT,None,<NA>,None,NaN,None,None,None,None
304,10000032,2180-06-27 18:49:00,encounter.end,DISCHARGE//HOSP//HOME,EVENT,None,<NA>,None,NaN,None,None,None,None
305,10000032,2180-08-07 17:50:00,encounter.end,DISCHARGE//HOSP//HOSPICE,EVENT,None,<NA>,None,NaN,None,None,None,None
306,10000032,2180-07-25 17:55:00,encounter.end,DISCHARGE//HOSP//HOME,EVENT,None,<NA>,None,NaN,None,None,None,None
307,10000068,2160-03-04 06:26:00,encounter.end,DISCHARGE//HOSP//nan,EVENT,None,<NA>,None,NaN,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
601,10004606,2159-03-06 16:51:00,encounter.end,DISCHARGE//HOSP//HOME,EVENT,None,<NA>,None,NaN,None,None,None,None
602,10004638,2173-06-12 16:11:00,encounter.end,DISCHARGE//HOSP//nan,EVENT,None,<NA>,None,NaN,None,None,None,None
603,10004638,2183-05-31 16:14:00,encounter.end,DISCHARGE//HOSP//nan,EVENT,None,<NA>,None,NaN,None,None,None,None
604,10004638,2171-12-21 15:40:00,encounter.end,DISCHARGE//HOSP//HOME,EVENT,None,<NA>,None,NaN,None,None,None,None


In [18]:
test_df['code'].unique()

array(['ADMIT//HOSP//URGENT', 'ADMIT//HOSP//EW EMER.',
       'ADMIT//HOSP//EU OBSERVATION', ..., 'GENDER//F', 'GENDER//M',
       'MEDS_DEATH'], shape=(2922,), dtype=object)

In [9]:
# path = "/data/padmalab_external/special_project/MIMIC_IV/MEDS_cohort/data/train/0.parquet"
path = '/data/padmalab_external/special_project/meds_pipeline_output/ahs/ahs_medicines_meds_core.parquet'
test_df = pd.read_parquet(path)
test_df.head()

,subject_id,time,event_type,code,code_system,value_num,unit,value_text,source_table,provenance_id
0,1,2008-04-01,medication.dispense,MEDICINE//ATC//A02BA02,ATC,60.0,TAB,day_supply=30 | ATC=A02BA02,PIN,1
1,1,2008-04-01,medication.dispense,MEDICINE//ATC//B01AA03,ATC,60.0,TAB,day_supply=30 | ATC=B01AA03,PIN,2
2,1,2008-04-01,medication.dispense,MEDICINE//ATC//B01AA03,ATC,60.0,TAB,day_supply=30 | ATC=B01AA03,PIN,3
3,1,2008-04-01,medication.dispense,MEDICINE//ATC//A02BA02,ATC,60.0,TAB,day_supply=30 | ATC=A02BA02,PIN,4
4,1,2008-04-18,medication.dispense,MEDICINE//ATC//G04CB01,ATC,30.0,TAB,day_supply=30 | ATC=G04CB01,PIN,5


In [13]:
# Filter rows where `code` starts with "DIAGNOSIS"
diag_df = test_df[test_df['code'].astype(str).str.startswith('DIAGNOSIS', na=False)]
print(f"Matches: {len(diag_df):,}")
diag_df[['subject_id', 'time', 'code']].head()

Matches: 17,662


,subject_id,time,code
383,10017886,2140-12-20 17:30:00,DIAGNOSIS//ICD//9//42843
384,10017886,2140-12-20 17:30:00,DIAGNOSIS//ICD//9//5990
385,10017886,2140-12-20 17:30:00,DIAGNOSIS//ICD//9//99664
386,10017886,2140-12-20 17:30:00,DIAGNOSIS//ICD//9//70711
387,10017886,2140-12-20 17:30:00,DIAGNOSIS//ICD//9//4280


In [29]:
# ls /data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_*

In [27]:
import pandas as pd
demographic_df = pd.read_pickle('/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_demographics.pkl')
# demographic_df = demographic_df.drop_duplicates('PATID')
demographic_df.head()

,PATID,SEX,Birth_Year
0,1,M,1949
1,1,M,1949
2,1,M,1949
3,1,M,1949
4,1,M,1949


In [22]:
import pandas as pd
death_df = pd.read_parquet('/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_death_20251124.parquet')
death_df.head()

,PATID,death_date,death_source
0,1,2021-04-24,vital_status
1,2,2020-10-15,vital_status
2,3,2019-03-09,vital_status
3,4,NaT,None
4,5,NaT,None


In [24]:
demographic_df = demographic_df.merge(death_df, left_on='PATID', right_on='PATID')
demographic_df.to_pickle('/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_demographics_with_death.pkl')

In [26]:
print (demographic_df.head())

   PATID SEX  Birth_Year death_date  death_source
0      1   M        1949 2021-04-24  vital_status
1      2   F        1936 2020-10-15  vital_status
2      3   F        1925 2019-03-09  vital_status
3      4   F        1978        NaT          None
4      5   M        1943        NaT          None


In [15]:
import pyreadstat
path = "/data/padmalab_external/special_project/AHS_Data_Release_2/rmt22884_reg_20211105.sas7bdat"
reg_df, meta = pyreadstat.read_sas7bdat(path, output_format="pandas")
reg_df.head()

,PATID,FYE,ACTIVE_COVERAGE,DEATH_IND,SEX,IN_MIGRATION_IND,OUT_MIGRATION_IND,PERS_REAP_END_DATE,PERS_REAP_END_RSN_CODE_GRP,Birth_Year
0,1.0,2007.0,1,0,M,0,0,NaT,N/A,1949.0
1,1.0,2008.0,1,0,M,0,0,NaT,N/A,1949.0
2,1.0,2009.0,1,0,M,0,0,NaT,N/A,1949.0
3,1.0,2010.0,1,0,M,0,0,NaT,N/A,1949.0
4,1.0,2011.0,1,0,M,0,0,NaT,N/A,1949.0


In [16]:
# demographic_df.drop_duplicates('PATID')
demographic_df['SEX'].unique()

array(['M', 'F', 'O'], dtype=object)

In [31]:
df = pd.read_csv('/data/padmalab_external/special_project/physionet.org/files/mimiciv/3.1/hosp/d_hcpcs.csv.gz', compression='gzip', low_memory=False)

In [33]:
df

,code,category,long_description,short_description
0,00000,NaN,NaN,Invalid Code
1,0001F,2.0,NaN,Composite measures
2,0002F,2.0,NaN,Composite measures
3,0003F,2.0,NaN,Composite measures
4,0004F,2.0,NaN,Composite measures
...,...,...,...,...
89203,XS,NaN,"Separate structure, a service that is distinct...",Separate organ/structure
89204,XU,NaN,"Unusual non-overlapping service, the use of a ...",Unusual separate service
89205,ZA,NaN,Novartis/sandoz,Novartis/sandoz
89206,ZB,NaN,Pfizer/hospira,Pfizer/hospira
